In [4]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Load the TCA-fused data
parquet_path = PROJECT_ROOT / "outputs" / "processed" / "sarawak_TCA_fused.parquet"
df = pd.read_parquet(parquet_path)
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month

# Feature Engineering (Must match xai_explainer.py exactly)
df['lulc_code'] = pd.Categorical(df['lulc_class']).codes
df['residual_bias'] = df['Gauge_Observed'] - df['P_merged_hier']

feature_cols = ['elev_m', 'lulc_code', 'month', 'precip_chirps', 'precip_era5_land', 'precip_persiann', 'P_merged_hier']
valid_mask = df[feature_cols + ['residual_bias']].notna().all(axis=1)
valid_df = df[valid_mask].copy()

print(f"✅ Loaded {len(valid_df)} valid samples.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Loaded 107949 valid samples.


In [5]:
# 1. Prepare Data
X = valid_df[feature_cols].values
y = valid_df['residual_bias'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define the models dictionary (This is the 'models' variable the loop needs!)
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'Neural Network (MLP)': MLPRegressor(hidden_layer_sizes=(50, 20), max_iter=1000, random_state=42)
}

# 3. Train, Evaluate, and SAVE every model
model_dir = PROJECT_ROOT / "outputs" / "processed"
model_dir.mkdir(parents=True, exist_ok=True)

results = []
for model_name, model_obj in models.items():
    print(f"🚀 Training {model_name}...")
    model_obj.fit(X_train, y_train)
    y_pred = model_obj.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': model_name, 'RMSE': rmse, 'R2': r2})
    
    # SAVE THE MODEL
    safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "")
    file_path = model_dir / f"model_{safe_name}.pkl"
    joblib.dump(model_obj, file_path)
    print(f"   ✅ Saved: {file_path}")

results_df = pd.DataFrame(results)
print("\n📊 Model Performance:")
print(results_df.to_string(index=False))

# Also save the best model separately for backward compatibility
best_model_name = results_df.loc[results_df['RMSE'].idxmin(), 'Model']
best_model_obj = models[best_model_name]
joblib.dump(best_model_obj, model_dir / "best_ml_model.pkl")
print(f"\n🏆 Best Model ({best_model_name}) also saved as best_ml_model.pkl")

🚀 Training Linear Regression...
   ✅ Saved: d:\my_project\outputs\processed\model_Linear_Regression.pkl
🚀 Training Random Forest...
   ✅ Saved: d:\my_project\outputs\processed\model_Random_Forest.pkl
🚀 Training Gradient Boosting...
   ✅ Saved: d:\my_project\outputs\processed\model_Gradient_Boosting.pkl
🚀 Training Neural Network (MLP)...
   ✅ Saved: d:\my_project\outputs\processed\model_Neural_Network_MLP.pkl

📊 Model Performance:
               Model       RMSE       R2
   Linear Regression 116.369240 0.054984
       Random Forest 111.843856 0.127055
   Gradient Boosting 110.912375 0.141535
Neural Network (MLP) 113.866008 0.095203

🏆 Best Model (Gradient Boosting) also saved as best_ml_model.pkl
